# 04 - Sparse Multitask CNN

**Purpose:** the softmax CNN predicts a valid composition vector, but still
assigns mass to absent materials. This notebook adds an explicit material-
presence head to reduce absent-material leakage.

A shared frozen EfficientNet-B0 feeds two heads: a **presence** head (which
materials are in the mixture) and a **composition** head (their proportions).
The fused prediction is `softmax(composition) * sigmoid(presence)`, renormalized
-- fully differentiable, no thresholding inside the model.


## 1. Imports and setup


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src import config, data, targets, splits, metrics, plots, models, training

config.ensure_output_dirs()
training.set_seed(config.RANDOM_SEED)
device = training.get_device()
print("device:", device)


## 2-5. Load metadata, build targets, validate


In [ ]:
metadata = data.load_metadata()
image_df = data.build_image_dataframe(metadata)

vocabulary = targets.build_material_vocabulary(metadata)
target_vectors = targets.build_targets(metadata, vocabulary=vocabulary)
materials = target_vectors["materials"]

report = data.validate_dataset(metadata)
assert report["ok"], report["problems"]
targets.validate_targets(target_vectors)
print(f"{len(metadata)} mixtures, {len(image_df)} images, materials = {materials}")


## 6. Grouped train / validation / test split by Code

The same grouped split and seed as Phase 2 and Phase 3, so all three models are
evaluated on exactly the same test mixtures.


In [ ]:
train_df, val_df, test_df = splits.grouped_train_val_test_split(
    image_df, group_col="Code",
    test_size=config.TEST_SIZE, val_size=config.VAL_SIZE, seed=config.RANDOM_SEED,
)
splits.check_no_leakage(train_df, val_df, test_df)
splits.check_groups_intact([train_df, val_df, test_df], image_df)
print("images -> train {}, val {}, test {}".format(len(train_df), len(val_df), len(test_df)))
print("codes  -> train {}, val {}, test {}".format(
    train_df["Code"].nunique(), val_df["Code"].nunique(), test_df["Code"].nunique()))


## 7. Targets: presence and volume fraction

- **presence** (multi-hot) is the material-presence classification target.
- **volume_fraction** is the composition target, as in Phase 3 (photographs
  reflect visible volume more directly than mass).


In [ ]:
presence_by_code = {c: target_vectors["presence"][i]
                    for i, c in enumerate(target_vectors["codes"])}
volume_by_code = {c: target_vectors["volume_fraction"][i]
                  for i, c in enumerate(target_vectors["codes"])}
mass_by_code = {c: target_vectors["mass_fraction"][i]
                for i, c in enumerate(target_vectors["codes"])}
density_vector = target_vectors["density_matrix"][0]


## 8. Datasets and dataloaders

Each photo is decoded once into memory (frozen backbone); the dataset returns
both the composition and presence targets per image.


In [ ]:
image_cache = training.preload_images(image_df, config.IMAGE_SIZE)

train_tf = training.build_transforms(train=True, image_size=config.IMAGE_SIZE)
eval_tf = training.build_transforms(train=False, image_size=config.IMAGE_SIZE)

def make_dataset(df, tf):
    return training.MultitaskImageDataset(
        df, volume_by_code, presence_by_code, tf,
        image_size=config.IMAGE_SIZE, image_cache=image_cache)

train_ds = make_dataset(train_df, train_tf)        # augmented, for training
train_eval_ds = make_dataset(train_df, eval_tf)    # deterministic, for prediction
val_ds = make_dataset(val_df, eval_tf)
test_ds = make_dataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=16, shuffle=False, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
print("batches -> train", len(train_loader), "val", len(val_loader), "test", len(test_loader))


## 9. Build the sparse multitask model (frozen EfficientNet-B0 + 2 heads)


In [ ]:
backbone, embedding_dim = models.build_efficientnet_backbone(pretrained=True, freeze=True)
model = models.SparseMultitaskCNN(backbone, embedding_dim, len(materials), dropout=0.3)
print("embedding dim:", embedding_dim)
print("trainable parameters (presence + composition heads):",
      models.count_trainable_parameters(model))


## 10. Train with the combined loss

`total = alpha * BCE(presence) + beta * KL(fused || volume) + gamma * mean(presence_prob)`

- **alpha = 1.0** -- presence classification (BCEWithLogits).
- **beta = 1.0** -- composition match of the *fused* output (KL divergence).
- **gamma = 0.01** -- a small sparsity penalty pushing presence probabilities
  down, so fewer materials are predicted present.


In [ ]:
ALPHA, BETA, GAMMA = 1.0, 1.0, 0.01
t0 = time.time()
model, history = training.train_multitask_model(
    model, train_loader, val_loader, device,
    epochs=30, lr=1e-3, patience=7, alpha=ALPHA, beta=BETA, gamma=GAMMA,
    save_path=config.MODEL_DIR / "sparse_multitask_cnn.pt",
)
train_seconds = time.time() - t0
best = history.loc[history["val_loss"].idxmin()]
print(f"training took {train_seconds:.0f}s over {len(history)} epochs")
print(f"best epoch {int(best['epoch'])}: val_loss {best['val_loss']:.4f}, "
      f"val_mae {best['val_mae']:.4f}, val_presence_f1 {best['val_presence_f1']:.4f}")


## 11. Training curves


In [ ]:
plots.plot_training_curve(history, save_path="04_training_curve.png")
plt.show()

# Presence-head F1 (validation) across epochs.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["epoch"], history["val_presence_f1"], marker="o", ms=3, color="#55A868")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation presence-head F1")
ax.set_title("Presence-head F1 over training")
plt.show()


## 12. Predict and verify fused composition vectors


In [ ]:
pred_train = training.predict_multitask_model(model, train_eval_loader, device, materials, density_vector, "train")
pred_val = training.predict_multitask_model(model, val_loader, device, materials, density_vector, "val")
pred_test = training.predict_multitask_model(model, test_loader, device, materials, density_vector, "test")

for df_ in (pred_train, pred_val, pred_test):
    Pm = df_[[f"pred_{m}" for m in materials]].to_numpy()
    Pv = df_[[f"pred_vol_{m}" for m in materials]].to_numpy()
    assert (Pm >= 0).all() and np.allclose(Pm.sum(1), 1.0, atol=1e-5)
    assert (Pv >= 0).all() and np.allclose(Pv.sum(1), 1.0, atol=1e-5)
print("fused predictions are valid compositions (non-negative, rows sum to 1)")


## 13. Evaluate image-level metrics (volume and mass space)


In [ ]:
def arrays(pred_df, true_prefix, pred_prefix):
    yt = pred_df[[f"{true_prefix}{m}" for m in materials]].to_numpy()
    yp = pred_df[[f"{pred_prefix}{m}" for m in materials]].to_numpy()
    return yt, yp

splits_named = [("train", pred_train), ("val", pred_val), ("test", pred_test)]
vol_metrics = {n: metrics.evaluate_composition(*arrays(d, "true_vol_", "pred_vol_"), materials)
               for n, d in splits_named}
mass_metrics = {n: metrics.evaluate_composition(*arrays(d, "true_", "pred_"), materials)
                for n, d in splits_named}

show = ["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]
print("Volume-space (image level):")
print(pd.DataFrame(vol_metrics).T[show].round(4).to_string())
print("\nMass-space (image level):")
print(pd.DataFrame(mass_metrics).T[show].round(4).to_string())


## 14. Mixture-level evaluation

Average the five fused image predictions per Code, renormalize, and score
against the Code-level target.


In [ ]:
def mixture_level(pred_df, value_cols, truth_by_code):
    tmp = pred_df[value_cols].copy()
    tmp.columns = materials
    tmp["Code"] = pred_df["Code"].values
    grouped = tmp.groupby("Code")[materials].mean()
    codes = grouped.index.tolist()
    pred_mix = training.renormalize_composition(grouped.to_numpy())
    y_true = np.array([truth_by_code[c] for c in codes])
    return codes, y_true, pred_mix

test_codes_mix, y_test_mix_mass, pred_test_mix_mass = mixture_level(
    pred_test, [f"pred_{m}" for m in materials], mass_by_code)
mix_mass_metrics = metrics.evaluate_composition(y_test_mix_mass, pred_test_mix_mass, materials)

_, y_test_mix_vol, pred_test_mix_vol = mixture_level(
    pred_test, [f"pred_vol_{m}" for m in materials], volume_by_code)
mix_vol_metrics = metrics.evaluate_composition(y_test_mix_vol, pred_test_mix_vol, materials)

print("test mixture-level (mass)  :", {k: round(mix_mass_metrics[k], 4) for k in show})
print("test mixture-level (volume):", {k: round(mix_vol_metrics[k], 4) for k in show})


## 15. Presence evaluation

Two presence F1 numbers, kept distinct so thresholds are never silently mixed:

- **composition-derived F1** (threshold 0.05 on the predicted fraction): this is
  what `metrics.evaluate_composition` reports for *every* model, so it is the
  fair cross-model number used in the comparison table.
- **presence-head F1** (threshold 0.5 on the explicit presence probability): a
  diagnostic specific to this model's presence head.


In [ ]:
true_pres = pred_test[[f"true_presence_{m}" for m in materials]].to_numpy()
pred_pres = (pred_test[[f"pred_presence_{m}" for m in materials]].to_numpy() > 0.5).astype(int)
presence_head_f1 = metrics.presence_f1(true_pres, pred_pres)

print("composition-derived presence F1 (thr 0.05, used in comparison):",
      round(mass_metrics["test"]["presence_F1"], 4))
print("presence-head F1 (thr 0.5, diagnostic)                        :",
      round(presence_head_f1, 4))


## 16. Compare against Random Forest and Softmax CNN

Same metric functions, same grouped split, same presence threshold. Mass-space,
mixture-level is the headline comparison.


In [ ]:
rf = pd.read_csv(config.TABLE_DIR / "baseline_random_forest_metrics.csv", index_col="split")
sm = pd.read_csv(config.TABLE_DIR / "transfer_learning_softmax_metrics.csv")
cols = ["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]

def sm_row(level):
    r = sm[(sm["split"] == "test") & (sm["level"] == level) & (sm["space"] == "mass")]
    return r[cols].iloc[0].to_dict()

comparison_df = pd.DataFrame([
    {"Model": "Random Forest",    "Level": "image",   "Space": "mass", **rf.loc["test", cols].to_dict()},
    {"Model": "Random Forest",    "Level": "mixture", "Space": "mass", **rf.loc["test_mixture", cols].to_dict()},
    {"Model": "Softmax CNN",      "Level": "image",   "Space": "mass", **sm_row("image")},
    {"Model": "Softmax CNN",      "Level": "mixture", "Space": "mass", **sm_row("mixture")},
    {"Model": "Sparse Multitask", "Level": "image",   "Space": "mass", **{k: mass_metrics["test"][k] for k in cols}},
    {"Model": "Sparse Multitask", "Level": "mixture", "Space": "mass", **{k: mix_mass_metrics[k] for k in cols}},
])[["Model", "Level", "Space"] + cols].round(4)
comparison_df


## 17. Save metrics and predictions


In [ ]:
rows = []
for n in ["train", "val", "test"]:
    rows.append({"split": n, "level": "image", "space": "volume", **vol_metrics[n]})
for n in ["train", "val", "test"]:
    rows.append({"split": n, "level": "image", "space": "mass", **mass_metrics[n]})
rows.append({"split": "test", "level": "mixture", "space": "mass", **mix_mass_metrics})
rows.append({"split": "test", "level": "mixture", "space": "volume", **mix_vol_metrics})
metrics_df = pd.DataFrame(rows)
metrics_path = config.TABLE_DIR / "sparse_multitask_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

# predict_multitask_model already returns every required column (mass, volume,
# presence prob, raw softmax), so the prediction tables are saved directly.
pred_all = pd.concat([pred_train, pred_val, pred_test], ignore_index=True)
pred_path = config.PREDICTION_DIR / "sparse_multitask_predictions.csv"
pred_all.to_csv(pred_path, index=False)
print("saved:", metrics_path.name, "|", pred_path.name, "| sparse_multitask_cnn.pt")


## 18. Plots


In [ ]:
yt_mass_test, yp_mass_test = arrays(pred_test, "true_", "pred_")
plots.plot_predicted_vs_true(yt_mass_test, yp_mass_test, materials,
                             save_path="04_pred_vs_true_test.png")
plt.show()

plots.plot_per_material_mae(metrics.per_material_mae(yt_mass_test, yp_mass_test, materials),
                            save_path="04_per_material_mae_test.png")
plt.show()

example_codes = test_codes_mix[:4]
plots.plot_example_predictions(image_df, example_codes, y_test_mix_mass[:4],
                               pred_test_mix_mass[:4], materials,
                               save_path="04_example_predictions_test.png")
plt.show()


## 19. Interpretation

Compare the three mass-space, mixture-level rows in the table above (Random
Forest -> Softmax CNN -> Sparse Multitask).

**Composition accuracy (MAE / R^2).** The presence head gates absent materials
to (near) zero. If this also matches or improves MAE and R^2 over the softmax
CNN, the explicit presence modelling helps; if MAE rises slightly, there is an
accuracy/sparsity trade-off to discuss.

**Presence F1.** Does the explicit presence head improve presence detection over
deriving presence from the softmax composition alone?

**False-positive absent mass.** This is the core question of Phase 4: softmax
left ~20% of predicted mass on absent materials. Multiplying by the presence
probability should push that lower -- the size of the drop measures how much the
presence head helps.

**Is the added complexity justified?** The model adds one extra linear head and
a two-term-plus-sparsity loss. Whether that is worthwhile depends on the leakage
reduction and accuracy change above.

**For the final comparison (notebook 05).** All three models now have saved,
comparably-computed metrics and predictions on the same grouped test split,
ready to be assembled into the final thesis comparison.
